In [ ]:
%%time
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from math import nan
from pathlib import Path
import dask
import yaml
import os
import warnings
import gc
import textwrap
import glob

from dask_jobqueue import PBSCluster
from dask.distributed import Client

warnings.filterwarnings('ignore')

In [ ]:
%%time
# Configure Dask to use a PBSCluster on Casper
dask.config.set({"distributed.scheduler.worker_saturation": 1.0})

cluster = PBSCluster(
    cores=1,
    memory='30GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    resource_spec='select=1:ncpus=1:mem=30GB',
    project='P04010022',
    walltime='24:00:00',
    interface='mgt',
    job_extra_directives=[
        "-o dask_logs/dask-worker.out",
        "-e dask_logs/dask-worker.err",
    ]
)

# Adjust number of workers as needed (start modestly to avoid hammering the filesystem)
cluster.scale(16)

# Dask dashboard URL for JupyterHub
dask.config.set({
    'distributed.dashboard.link':
    'https://jupyterhub.hpc.ucar.edu/stable/user/{USER}/proxy/{port}/status'
})

client = Client(cluster)
client

In [ ]:
cluster

In [ ]:
%%time
# CAM HR input files
h5a = sorted(glob.glob(
    "/glade/campaign/cesm/community/amwg/cam_hr/"
    "f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/*.h5a.*"
))

h5i = sorted(glob.glob(
    "/glade/campaign/cesm/community/amwg/cam_hr/"
    "f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/*.h5i.*"
))

print("Number of h5a files:", len(h5a))
print("Number of h5i files:", len(h5i))

if len(h5a) != len(h5i):
    raise ValueError("h5a and h5i file lists have different lengths!")

In [ ]:
%%time
import os, gc
import numpy as np
import xarray as xr

# Constants for Tb calculation
sigma = 5.67e-8        # W m-2 K-4
a = 1.228
b = -1.106e-3          # K-1

h5avars = ['PRECT']

outdir = "/glade/derecho/scratch/hannay/MOAPP/daily_files/"
os.makedirs(outdir, exist_ok=True)

def process_one_pair(ifile):
    """
    Process a single (h5a, h5i) pair into daily
    CAM7_L58_HR_hourly.YYYY-MM-DD.nc files (24 hourly timesteps each),
    with PRECT (mm/hr), Tb, and 2D X/Y, and unlimited time.
    """

    print("=====")
    #print(h5a[ifile])
    #print(h5i[ifile])

    fname = os.path.basename(h5i[ifile])

    # --- Open h5a and convert PRECT (m/s -> mm/hr) ---
    h5adat = xr.open_dataset(h5a[ifile], chunks={"time": 24}, engine="netcdf4")
    h5adat = h5adat[h5avars]  # keep only PRECT

    # Keep original attrs and update units
    prect_attrs = h5adat["PRECT"].attrs
    h5adat["PRECT"] = (h5adat["PRECT"] * 1000.0 * 3600.0).astype("float32")
    h5adat["PRECT"].attrs = prect_attrs
    h5adat["PRECT"].attrs["units"] = "mm hr-1"
    h5adat["PRECT"].encoding["_FillValue"] = np.float32(np.nan)

    # --- Open h5i and compute Tb from FLUT ---
    h5idat = xr.open_dataset(h5i[ifile], chunks={"time": 24}, engine="netcdf4")

    FLUT = h5idat["FLUT"]

    # Sanity check: FLUT should not be all zero
    if float(FLUT.max()) == 0.0:
        raise RuntimeError(f"FLUT is all zero in {fname}, cannot compute Tb.")

    Tf = (FLUT / sigma) ** 0.25
    Tb = (-a + np.sqrt(a**2 + 4.0 * b * Tf)) / (2.0 * b)

    Tb = Tb.rename("Tb")
    Tb.attrs["long_name"] = "flux equivalent brightness temperature"
    Tb.attrs["units"] = "K"
    h5idat["Tb"] = Tb

    # Use h5i time axis for merged output
    h5adat["time"] = h5idat.time.values

    # --- Merge everything (this matches your working version) ---
    newdat = xr.merge([h5adat, h5idat])

    # --- Add 2D X/Y as auxiliary variables, WITHOUT renaming dims ---

    # We assume standard CAM layout with 1D lat, lon coords
    if "lat" not in newdat.coords or "lon" not in newdat.coords:
        raise KeyError("Expected 1D 'lat' and 'lon' coordinates in merged dataset.")

    lat_1d = newdat["lat"]  # dims: (lat)
    lon_1d = newdat["lon"]  # dims: (lon)

    # Only add if not already present
    if "Y" not in newdat.variables or "X" not in newdat.variables:
        # Broadcast to 2D (lat, lon)
        Y_2d, X_2d = xr.broadcast(lat_1d, lon_1d)
        Y_2d = Y_2d.rename("Y")
        X_2d = X_2d.rename("X")

        Y_2d.attrs["long_name"] = "latitude on (lat,lon) grid"
        Y_2d.attrs["units"] = lat_1d.attrs.get("units", "degrees_north")

        X_2d.attrs["long_name"] = "longitude on (lat,lon) grid"
        X_2d.attrs["units"] = lon_1d.attrs.get("units", "degrees_east")

        newdat["Y"] = Y_2d
        newdat["X"] = X_2d

    # --- Daily splitting: one file per day, 24 timesteps each ---
    written_files = []

    time = newdat["time"]
    # CFTime-friendly: floor to day
    time_floor = time.dt.floor("D")
    unique_days = np.unique(time_floor.values)

    for d in unique_days:
        # d is cftime.Datetime*
        date_str = f"{d.year:04d}-{d.month:02d}-{d.day:02d}"
        outpath_day = os.path.join(outdir, f"CAM7_L58_HR_hourly.{date_str}.nc")

        if os.path.exists(outpath_day):
            print(f"File already exists, skipping: {outpath_day}")
            continue

        print(f"  Writing daily file: {outpath_day}")

        # Boolean mask for this specific day
        mask = time_floor == d
        day_ds = newdat.isel(time=mask)

        # Final sanity check: Tb not all zero for this day
        if float(day_ds["Tb"].max()) == 0.0:
            raise RuntimeError(
                f"Tb is zero in daily slice {date_str} – something is wrong upstream."
            )

        # Write with time as an unlimited dimension
        day_ds.to_netcdf(outpath_day, format="NETCDF4", unlimited_dims="time")

        written_files.append(outpath_day)

    # --- Cleanup ---
    newdat.close()
    h5adat.close()
    h5idat.close()

    del newdat, h5adat, h5idat, FLUT, Tf, Tb
    gc.collect()

    return written_files


In [ ]:
%%time 
import os, gc
import numpy as np
import xarray as xr

# Constants for Tb calculation
sigma = 5.67e-8        # W m-2 K-4
a = 1.228
b = -1.106e-3          # K-1

h5avars = ['PRECT']

outdir = "/glade/derecho/scratch/hannay/MOAPP/daily_files/"
os.makedirs(outdir, exist_ok=True)


def process_one_pair(ifile):
    """
    For one (h5a, h5i) pair:
      - convert PRECT to mm/hr
      - compute Tb from FLUT
      - merge everything
      - then write one daily file per 24 timesteps:
        CAM7_L58_HR_hourly.YYYY-MM-DD.nc
      with time as UNLIMITED.
    """

    print("=====")
    #print("h5a:", h5a[ifile])
    #print("h5i:", h5i[ifile])

    # --- Open h5a and convert PRECT (exactly like your working code) ---
    h5adat = xr.open_dataset(h5a[ifile], chunks={"time": 24}, engine="netcdf4")
    h5adat = h5adat[h5avars]

    prect_attrs = h5adat["PRECT"].attrs
    h5adat["PRECT"] = h5adat["PRECT"] * 1000.0 * 3600.0  # mm/hr
    h5adat["PRECT"].attrs = prect_attrs
    h5adat["PRECT"].attrs["units"] = "mm hr-1"

    # --- Open h5i and compute Tb from FLUT (same as working code) ---
    h5idat = xr.open_dataset(h5i[ifile], chunks={"time": 24}, engine="netcdf4")
    FLUT = h5idat["FLUT"]

    print("  FLUT min/max:", float(FLUT.min()), float(FLUT.max()))

    Tf = (FLUT / sigma) ** 0.25
    Tb = (-a + np.sqrt(a**2 + 4.0 * b * Tf)) / (2.0 * b)

    Tb = Tb.rename("Tb")
    Tb.attrs["long_name"] = "flux equivalent brightness temperature"
    Tb.attrs["units"] = "K"

    # Make sure Tb has a clean encoding (no weird inherited scaling)
    Tb = Tb.astype("float32")
    Tb.encoding = {}
    Tb.encoding["_FillValue"] = np.float32(np.nan)

    h5idat["Tb"] = Tb

    print("  Tb TEST min/max:", float(Tb.min()), float(Tb.max()))

    # --- Merge exactly like your original script ---
    h5adat["time"] = h5idat.time.values
    newdat = xr.merge([h5adat, h5idat])

    print("  Tb after merge min/max:",
          float(newdat["Tb"].min()), float(newdat["Tb"].max()))

    # --- Split into daily chunks by index: 0–23, 24–47, ... ---
    written_files = []

    nt = newdat.dims["time"]
    if nt % 24 != 0:
        raise RuntimeError(f"Time length {nt} is not a multiple of 24.")

    ndays = nt // 24
    print(f"  Found {nt} timesteps -> {ndays} days of 24 hours")

    for day_idx in range(ndays):
        t0 = day_idx * 24
        t1 = t0 + 24

        day_ds = newdat.isel(time=slice(t0, t1))

        # use first time as date
        t0_val = day_ds["time"].values[0]
        date_str = f"{t0_val.year:04d}-{t0_val.month:02d}-{t0_val.day:02d}"

        outpath_day = os.path.join(outdir, f"CAM7_L58_HR_hourly.{date_str}.nc")

        # Important: if you’re debugging, consider deleting old files first,
        # or force overwrite here.
        if os.path.exists(outpath_day):
            print(f"  File already exists, skipping: {outpath_day}")
            continue

        tb_min = float(day_ds["Tb"].min())
        tb_max = float(day_ds["Tb"].max())
        print(f"    Day {date_str}: Tb min/max before write = {tb_min}, {tb_max}")

        if tb_max == 0.0 and tb_min == 0.0:
            raise RuntimeError(f"Tb is all zero in daily slice {date_str} BEFORE writing.")

        # Write with time as unlimited
        day_ds.to_netcdf(outpath_day, format="NETCDF4", unlimited_dims="time")

        written_files.append(outpath_day)

    # --- Cleanup ---
    newdat.close()
    h5adat.close()
    h5idat.close()
    del newdat, h5adat, h5idat, FLUT, Tf, Tb
    gc.collect()

    return written_files


In [ ]:
%%time
# Optional: test a single file before running in parallel
_ = process_one_pair(0)

In [ ]:
%%time
from dask import delayed
from dask.distributed import wait

nfiles = len(h5a)
print("Total files:", nfiles)

# For testing, you can restrict indices, e.g. range(10)
indices = range(nfiles)

tasks = [delayed(process_one_pair)(i) for i in indices]

futures = client.compute(tasks)
wait(futures)

print("All requested files processed.")

In [ ]:
cluster.close()
client.close()
